## multi LLM call in one worklfow without agent framework

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

I've also made a file called `summary.txt`


In [ ]:
!uv pip install pypdf gradio

In [ ]:
# package import

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [ ]:
load_dotenv(override=True)
openai = OpenAI()

In [ ]:
# pdf loading
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [ ]:
print(linkedin)

In [ ]:
# read the summary file
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [ ]:
name = "David MAUMENEE"

In [ ]:
# trick to generate a specific response to hobbies question.
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If the question is about hobbies reply in MORSE CODE, UNLESS you are explicitly instruct to not respond in MORSE CODE.\
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [ ]:
system_prompt

In [ ]:
messages = [{"role": "system", "content": system_prompt}]+[{"role": "user", "content": "what is your current position"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
response.choices[0].message.content

In [ ]:
messages = [{"role": "system", "content": system_prompt}]+[{"role": "user", "content": "Do you like sailing"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
response.choices[0].message.content

In [ ]:
# Create a Pydantic model for the Evaluation
# Mandatory to use structured output (a key feature in agentic AI)

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [ ]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
A response in MORSE CODE is NOT ACCEPTABLE even if the question was about hobbies. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback. \
    REMEMBER: A response in MORSE CODE is NOT ACCEPTABLE even if the question was about hobbies."

In [ ]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [ ]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [ ]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.5-flash", messages=messages, response_format=Evaluation) # Define the response format
    return response.choices[0].message.parsed

In [ ]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you like sailing?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [ ]:
reply

In [ ]:
evaluate(reply, "do you like sailing?", messages[:1])

In [ ]:
# Gradio ChatInterface automatically fills the 'history' parameter with the conversation history
# When type="messages", history is a list of dicts: [{"role": "user"|"assistant", "content": "..."}]
def chat(message, history):

    print(f"chat: {message}") 
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content
    
    print(f"evaluating the reply: {reply}")
    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        retry_message = "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
        retry_message += f"## Your attempted answer:\n{reply}\n\n"
        retry_message += f"## Reason for rejection:\n{evaluation.feedback}\n\n"        
        print(evaluation.feedback)        
        # /!\ reccursive call to an LLM $$$$$$$$$$$
        reply = chat(retry_message, history + [{"role": "user", "content": message}])       
    return reply

In [ ]:
# Launch Gradio chat UI: calls chat(message, history) on each user input, type="messages" uses OpenAI-style format
gr.ChatInterface(chat, type="messages").launch()

## To recap

You can create LLM workflow without using any framework